In [1]:
# LangChain + LangGraph 협업에 대한 이해
# 시나리오 : 회사 내 문서 + 일반 문서 둘 다 답해야 하는 챗봇
# 질문이 회사 문서 관련이면 RAG로 문서 검색 후 답변
# 질문이 일반 문서 관련이라면 그냥 LLM으로 답변하기ㅏ
# 이 로직을 LangGraph가 흐름을 제어(WorkFlow)하고, 각 단계에서 사용하는 LLM/RAG 체인은 LangChain이 담당
!pip install -U langgraph langchain langchain-community chromadb langchain-openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.1/157.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 32.1 MB/s eta 0:00:0

In [13]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is not set")

llm = ChatOpenAI(model_name="gpt-4o-mini", openai_api_key=OPENAI_API_KEY, temperature=0.3)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY
)

# 사내 문서 예 - vectorDB에 저장
docs = [
    Document(page_content="우리 회사의 정식 근무시간은 오전 9시부터 오후 6시까지이다"),
    Document(page_content="연차 휴가는 1년에 15일이 기본 제공되며, 2년차부터 하루씩 증가한다"),
    Document(page_content="사내 메신저는 슬랙을 사용하며, 중요 공지는 #notice 채널을 이용한다"),
    Document(page_content="아침, 점심, 저녁 끼니와 기숙사는 무료로 제공한다")
]

vectorstore = Chroma.from_documents(
    docs,
    embedding = embeddings,
    collection_name="company-col",
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
  return "\n\n".join(d.page_content for d in docs)

# RAG 체인 정의 : 사내 문서 관련 질문 prompt
RAG_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 회사 내부 문서를 기반으로 답하는 어시스턴트야\n" +
            "다음 문서 내용을 근거로 사용자의 질문에 한국어로 답해줘.\n\n" +
            "문서내용:\n{context}"
        ),
        ("human","질문:{question}")
    ]
)

# LCEL 구성
rag_chain = (
    {
        "context":retriever | format_docs,
        "question":RunnablePassthrough(),
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 분류 체인 정의 : 이 질문이 "사내" 관련인지, 아니면 "일반"인지 판단
CLASSIFY_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 사용자의 질문이 '사내 정책/회사 규칙 등' 관련 질문인지\n" +
            "'일반 상식' 질문이지 분류하는 분류 전문가야\n" +
            "답변은 반드시 'rag' 또는 'llm' 둘 중 하나로 반드시 출력해\n" +
            "- 사내 근무시간, 연차, 사내 메신저, 사내 규정, 복지 등은 'rag'\n" +
            "- 날씨, 역사, 스포츠, 과학 일반 상식 등은 'llm'"
        ),
        ("human","질문:{question}")
    ]
)

classify_chain = CLASSIFY_PROMPT | llm | StrOutputParser()

# 일반 LLM 답변 체인 정의 : 이 질문이 '일반 상식'일 때 prompt
LLM_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 일반 상식 질문에 친절하게 답하는 챗봇이야"
        ),
        ("human","질문:{question}")
    ]
)
llm_chain = LLM_PROMPT | llm | StrOutputParser()


In [14]:
# LangGraph 구조 정의
class ChatState(TypedDict):
    question: str
    route: Literal["rag", "llm", "unknown"]
    answer: str

def classify_node(state: ChatState) -> ChatState:
  q = state["question"]
  result = classify_chain.invoke({"question": q}).strip().lower()
  print(f"result : {result}")

  if "rag" in result:
    route = "rag"
  elif "llm" in result:
    route = "llm"
  else:
    route = "unknown"

  return {"route": route}

# RAG 기반으로 사내 문서로 답변
def rag_answer_node(state: ChatState) -> ChatState:
  q = state["question"]
  answer = rag_chain.invoke(q)
  print(f"[rag_answer_node] RAG 답변 : {answer}")
  return {"answer": answer}

# 일반 상식에 대한 LLM 답변
def llm_answer_node(state: ChatState) -> ChatState:
  q = state["question"]
  answer = llm_chain.invoke({"question": q})
  return {"answer": answer}

# LLM이 rag, llm 이외의 unknown 답을 준 경우 대비책
def fallback_node(state: ChatState) -> ChatState:
  # 분류가 애매한 경우
  q = state["question"]
  answer = (
      "질문이 사내 관련 내용인지, 일반 상식인지 애매하므로 일반 답변할게요.\n" +
      llm_chain.invoke({"question":q})
  )
  return {"answer": answer}

In [16]:
# LangChain 그래프 구성
graph_builder = StateGraph(ChatState)

graph_builder.add_node("classify", classify_node)
graph_builder.add_node("rag_answer", rag_answer_node)
graph_builder.add_node("llm_answer", llm_answer_node)
graph_builder.add_node("fallback", fallback_node)

graph_builder.set_entry_point("classify")

# 분기 로직
def route_decider(state: ChatState) -> str:
  route = state["route"]
  return route

graph_builder.add_conditional_edges(
    "classify",
    route_decider,
    {
        "rag":"rag_answer",
        "llm":"llm_answer",
        "unknown":"fallback"
    }
)

graph_builder.add_edge("rag_answer", END)
graph_builder.add_edge("llm_answer", END)
graph_builder.add_edge("fallback", END)

app = graph_builder.compile()

if __name__=="__main__":
  while True:
    user_q = input("\n질문(종료:q)")         # 예 : 근무시간은
    if user_q.lower() == "q":
      print("================[종료]==================")
      break

    state:ChatState = {
        "question":user_q,
        "route":"unknown",
        "answer":""
    }

    result = app.invoke(state)
    print(f"최종답변 : {result['answer']}")


질문(종류:q)근무시간에 대해서 알려줘
result : rag
[rag_answer_node] RAG 답변 : 우리 회사의 정식 근무시간은 오전 9시부터 오후 6시까지입니다.
최종답변 : 우리 회사의 정식 근무시간은 오전 9시부터 오후 6시까지입니다.

질문(종류:q)q
================[종료]==================
